# 01 — Data Audit

Goal: inspect the original Kaggle `listings.csv` without modifying it. The audit establishes grain, integrity, missingness, cardinality and deterministic parsing feasibility before cleaning.

In [ ]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "listings.csv"
df = pd.read_csv(RAW_PATH, low_memory=False)
df.shape

## Grain and uniqueness

The expected grain is one row per Airbnb listing. `id` is tested as the listing identifier.

In [ ]:
print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Unique listing IDs:", df["id"].nunique())
print("Duplicate listing IDs:", df["id"].duplicated().sum())
print("Fully duplicated rows:", df.duplicated().sum())

## Raw-file integrity

In [ ]:
sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
print("SHA-256:", sha256)
print("last_scraped values:")
print(df["last_scraped"].value_counts(dropna=False))

## Schema

In [ ]:
schema = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().sum(), "unique": df.nunique(dropna=True)})
schema

## Highest missingness

In [ ]:
missing = (df.isna().mean().mul(100).sort_values(ascending=False).to_frame("missing_pct"))
missing.head(20)

## Completely empty fields

In [ ]:
all_null = [c for c in df.columns if df[c].isna().all()]
all_null

## Dashboard-friendly category cardinality

In [ ]:
for col in ["room_type", "cancellation_policy", "property_type", "neighbourhood_cleansed", "bed_type"]:
    print(f"{col}: {df[col].nunique(dropna=True)} unique values")
    print(df[col].value_counts(dropna=False).head(30), "\n")

## Numeric conversion checks

These checks do not modify the raw dataframe. They verify whether currency and percentage strings can be converted without losing non-null source values.

In [ ]:
def parse_currency(series):
    return pd.to_numeric(series.astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False), errors="coerce")

def parse_pct(series):
    return pd.to_numeric(series.astype("string").str.replace("%", "", regex=False), errors="coerce")

for col in ["price", "cleaning_fee", "extra_people"]:
    parsed = parse_currency(df[col])
    failed = ((df[col].notna()) & parsed.isna()).sum()
    print(col, "non-null source:", df[col].notna().sum(), "parse failures:", failed, "range:", (parsed.min(), parsed.max()))

for col in ["host_response_rate", "host_acceptance_rate"]:
    parsed = parse_pct(df[col])
    failed = ((df[col].notna()) & parsed.isna()).sum()
    print(col, "non-null source:", df[col].notna().sum(), "parse failures:", failed)

## Audit decision

- No duplicated listing IDs.
- No fully duplicated rows.
- No row deletion is justified.
- Missing values will remain missing.
- Extreme prices remain because this file alone does not prove they are invalid.
- Cleaning will select analysis-relevant columns and perform deterministic type conversions only.